Realizamos las importaciones necesarias:

In [ ]:
import os
import csv
import ast
import re
import time
import nltk
import collections
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support, f1_score
from sklearn.preprocessing import rlBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from joblib import parallel_backend

Fijamos las variables globales del cuaderno:

In [234]:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "etiquetas_test.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

Definimos funciones de carga de datos, tokenización, y cálculo de métricas y estadísticas:

In [240]:
# Función para cargar el corpus:
def load_corpus(filename, delimiter=','):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Creamos un set para los títulos ya encontrados a fin de evitar duplicados:
        titles = set()

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            if row[0] not in titles:
                titles.add(row[0])
            else:
                continue

            papers.append((row[0], row[1]))
            terms.append(ast.literal_eval(row[-1]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    probs = {}
    for label, freq in sorted(ct.items(), key=lambda item: item[1], reverse=True):
        prob = freq / len(texts)
        probs[label] = prob
        print(f"{label}: {freq} ({prob:.1%})", end="  ")
    print("\n")
    return probs


try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

Cargamos el dataset de entrenamiento y reservamos una partición de validación:

In [ ]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = rlBinarizer()
y_bin = mlb.fit_transform(y_all)
print(mlb.classes_)

# Tokenizamos los títulos y los resúmenes:
raw_tok = [tokenize(title) + " " + tokenize(summary) for title, summary in raw_all]

# Creamos una partición aleatoria de entrenamiento/desarrollo:
train_tok, dev_tok, y_train, y_dev = train_test_split(raw_tok, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

Papers de entrenamiento: 32570
['cs.AI' 'cs.CL' 'cs.CR' 'cs.CV' 'cs.CY' 'cs.DC' 'cs.GR' 'cs.HC' 'cs.IR'
 'cs.IT' 'cs.LG' 'cs.MA' 'cs.MM' 'cs.NE' 'cs.RO' 'cs.SI' 'cs.SY' 'eess.IV'
 'eess.SP' 'eess.SY' 'math.IT' 'math.OC' 'math.ST' 'physics.chem-ph'
 'stat.AP' 'stat.ME' 'stat.ML' 'stat.TH']


Comprobamos la distribución de clases entre el conjunto de entrenamiento y el de validación:

In [241]:
labels_train = [cat for paper_cats in mlb.inverse_transform(y_train) for cat in paper_cats]
labels_dev = [cat for paper_cats in mlb.inverse_transform(y_dev) for cat in paper_cats]

prob_c_train = show_stats(train_tok, labels_train, "Entrenamiento")
prob_c_dev = show_stats(dev_tok, labels_dev, "Validación")

Entrenamiento
-------------
Muestras: 27684   Tokens: 4836492   Tokens por muestra: 174.70
cs.CV: 16786 (60.6%)  cs.LG: 14608 (52.8%)  stat.ML: 8142 (29.4%)  cs.AI: 3819 (13.8%)  eess.IV: 1071 (3.9%)  cs.RO: 894 (3.2%)  cs.CL: 609 (2.2%)  cs.NE: 431 (1.6%)  cs.CR: 235 (0.8%)  cs.SI: 219 (0.8%)  math.OC: 219 (0.8%)  cs.GR: 215 (0.8%)  eess.SP: 198 (0.7%)  cs.MM: 178 (0.6%)  cs.MA: 145 (0.5%)  cs.IR: 117 (0.4%)  cs.DC: 72 (0.3%)  cs.CY: 67 (0.2%)  cs.SY: 57 (0.2%)  eess.SY: 57 (0.2%)  stat.ME: 42 (0.2%)  stat.AP: 40 (0.1%)  cs.IT: 37 (0.1%)  math.IT: 37 (0.1%)  math.ST: 35 (0.1%)  stat.TH: 35 (0.1%)  cs.HC: 35 (0.1%)  physics.chem-ph: 19 (0.1%)  

Validación
----------
Muestras: 4886   Tokens: 856911   Tokens por muestra: 175.38
cs.CV: 2996 (61.3%)  cs.LG: 2554 (52.3%)  stat.ML: 1422 (29.1%)  cs.AI: 677 (13.9%)  eess.IV: 190 (3.9%)  cs.RO: 161 (3.3%)  cs.CL: 95 (1.9%)  cs.NE: 78 (1.6%)  cs.SI: 44 (0.9%)  cs.GR: 44 (0.9%)  math.OC: 40 (0.8%)  cs.CR: 39 (0.8%)  cs.MA: 32 (0.7%)  cs.MM: 31 

## Clasificadores *dummy*

Probamos dos clasificadores *dummy* uno que no predice ninguna clase —rellenando a ceros la matriz de predicciones— y otro que predice todas para cada *paper* (matriz a unos):

In [157]:
print("== Matriz a ceros:")
print(compute_metrics(y_dev, np.zeros_like(y_dev)))
print()

print("== Matriz a unos:")
print(compute_metrics(y_dev, np.ones_like(y_dev)))

== Matriz a ceros:
{'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'hamming_loss': 0.06245248815858722}

== Matriz a unos:
{'accuracy': 0.0, 'f1': 0.08929290413057454, 'precision': 0.06245248815858722, 'recall': 1.0, 'hamming_loss': 0.9375475118414128}


**Resultados:**
* La *Hamming loss* del primer clasificador es muy baja. Esto es debido a que la gran mayoría de entradas en la matriz dispersa de las etiquetas son ceros, por lo que el valor de la pérdida responde exclusivamente a la proporción de unos que hay en ella (un 6.24 %), que es a su vez la precisión del segundo clasificador.

* El *recall* del segundo clasificador es de 1, el máximo posible, pues, al haber predicho todas las clases como positivas en todos los *papers*, resulta evidente que todos sus positivos son a la vez positivos reales.

## Regresión logística (RL)

Definimos el vectorizador TF-IDF para generar las matrices dispersas de los conjuntos de entrenamiento y validación:

In [158]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

# Ordenamos las entradas de cada fila y sumamos posibles duplicados
# para evitar fallos en la paralelización de la RL:
A.sort_indices()
A.sum_duplicates()
B.sort_indices()

### Parámetros del clasificador

Entrenamos una regresión logística iterando sobre distintos valores de C, comprendidos entre 0.1 y 10:

In [ ]:
rl = {}
for c in [0.1, 0.25, 0.5, 0.75, 1, 2, 3, 5, 7, 10]:
    for cw in [None, "balanced"]:
        
        rl[(c, cw)] = OneVsRestClassifier(
            LogisticRegression(
                solver="liblinear",
                C=c,
                class_weight=cw,
                random_state=SEED
            ),
            n_jobs=-1
        ).fit(A, y_train)

        y_pred = rl[(c, cw)].predict(B)
        print(f"=== C={c}" + (', class_weight="balanced"' if cw else ""))
        print(compute_metrics(y_dev, y_pred))
        print()

=== C=0.1
{'accuracy': 0.4715513712648383, 'f1': 0.08477958582519562, 'precision': 0.10624308213948942, 'recall': 0.08095884342545283, 'hamming_loss': 0.02695017835214315}

=== C=0.1, class_weight="balanced"
{'accuracy': 0.20773638968481375, 'f1': 0.2529507324718815, 'precision': 0.19300823759942465, 'recall': 0.5039863639746961, 'hamming_loss': 0.05178059762586983}

=== C=0.25
{'accuracy': 0.4926320098239869, 'f1': 0.09298136057809189, 'precision': 0.1118685281808026, 'recall': 0.08830748937393199, 'hamming_loss': 0.025875679784807905}

=== C=0.25, class_weight="balanced"
{'accuracy': 0.25542365943512074, 'f1': 0.2711067979652536, 'precision': 0.20980719871998996, 'recall': 0.48201238088841913, 'hamming_loss': 0.04585989123443073}

=== C=0.5
{'accuracy': 0.49672533769954974, 'f1': 0.10004177370049039, 'precision': 0.15122167820736201, 'recall': 0.09343684494356076, 'hamming_loss': 0.025473656511315126}

=== C=0.5, class_weight="balanced"
{'accuracy': 0.2992222677036431, 'f1': 0.274624

**Resultados:**
* Equilibrar los pesos de las clases se posiciona claramente como el camino a seguir. Esto deriva de la heterogeneidad en la distribución de clases: `cs.CV` y `cs.LG` aparecen en más del **50 %** de papers, mientras que otras como `stat.ME` solo están asignadas a un **0.1 %** de las muestras. El modelo generalmente tenderá a evitar predecir clases así de pequeñas si toman pesos proporcionales a su tamaño.

* Los resultados parecen alcanzar su máximo con valores de `C` entre 2 y 5, con una F1 macro de **0.2978** para `C=3` y `class_weight="balanced"`.

Partiendo de los resultados del experimento anterior, a continuación vamos a probar valores de `C` en el intervalo [2, 5], fijando `class_weight` a `"balanced"`:

In [ ]:
for c in np.arange(2, 5, 0.25):

    rl[(c, "balanced")] = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear", 
            C=c, 
            class_weight="balanced",
            random_state=SEED
        ),
        n_jobs=-1
    ).fit(A, y_train)

    y_pred = rl[(c, "balanced")].predict(B)
    print(f"=== C={c}" + (', class_weight="balanced"' if cw else ""))
    print(compute_metrics(y_dev, y_pred))
    print()

=== C=2.0, class_weight="balanced"
{'accuracy': 0.37392550143266473, 'f1': 0.29357611401087785, 'precision': 0.2496512319226008, 'recall': 0.38329430027838934, 'hamming_loss': 0.03499795333606222}

=== C=2.25, class_weight="balanced"
{'accuracy': 0.3790421612771183, 'f1': 0.29407630020884973, 'precision': 0.252379285299932, 'recall': 0.3774436711006074, 'hamming_loss': 0.03456669200631542}

=== C=2.5, class_weight="balanced"
{'accuracy': 0.3823168235775686, 'f1': 0.296863675678225, 'precision': 0.2590495092917331, 'recall': 0.3732341926554747, 'hamming_loss': 0.03421583533126718}

=== C=2.75, class_weight="balanced"
{'accuracy': 0.38620548505935326, 'f1': 0.2983685131424023, 'precision': 0.26344627966727835, 'recall': 0.3699911537293408, 'hamming_loss': 0.03388690719840945}

=== C=3.0, class_weight="balanced"
{'accuracy': 0.39111747851002865, 'f1': 0.2978241145053749, 'precision': 0.26443823651433523, 'recall': 0.3651350958451381, 'hamming_loss': 0.03356528857961523}

=== C=3.25, class

Para afinar incluso más la búsqueda, vamos a comprobar los valores que rodean a 2.75 y quedarnos con el mejor:

In [ ]:
max_f1_rl = -1
for c in np.arange(2.65, 2.85, 0.01):
    c = round(c, 2)

    rl[(c, "balanced")] = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",
            C=c,
            class_weight="balanced",
            random_state=SEED
        ),
        n_jobs=-1
    ).fit(A, y_train)
    
    y_pred = rl[(c, "balanced")].predict(B)
    # print(f'=== C={c}, class_weight="balanced"')
    f1_rl = compute_metrics(y_dev, y_pred)['f1']
    if f1_rl > max_f1_rl:
        max_f1_rl = f1_rl
        best_c_rl = c

print("Mejor C:", best_c_rl)
print("Mejor F1 macro:", max_f1_rl)

Mejor C: 2.82
Mejor F1 macro: 0.29896037773550976


Para la mejor C que hemos obtenido, optimizaremos los **umbrales de decisión** de cada clase. Esto es, la probabilidad que necesita la clase en el *OneVsRestClassifier* para ser elegida. Este ajuste es especialmente importante para clases con muy pocas muestras:

In [339]:
# Definimos una función para ajustar los umbrales de decisión de un clasificador:
def adjust_thresholds(model, X, min_th=0.01, max_th=1, n_th=200, alg="rl"):
    # Obtenemos las probabilidades para cada clase que ha asociado el modelo a cada paper:
    scores_dev = model.predict_proba(X) if alg == "rl" else model.decision_function(X)

    # Creamos un vector con tantos elementos como clases, inicializado a ceros:
    n_classes = len(mlb.classes_)
    optimal_thresholds = np.zeros(n_classes)

    # Definimos una lista de 'n_th' umbrales de decisión equiespaciados entre 'min_th' y 'max_th':
    thresholds = np.linspace(min_th, max_th, n_th)

    # Iteramos sobre cada clase:
    for c in range(n_classes):
        max_f1_c = -1
        best_th = 0

        # Obtenemos la columna de etiquetas y probabilidades para cada paper asociadas a la clase:
        y_dev_c = y_dev[:, c]
        scores_c = scores_dev[:, c]

        # Iteramos sobre cada umbral:
        for th in thresholds:
            # Obtenemos las predicciones de etiquetas asociadas a ese umbral y calculamos su F1 individual:
            preds_c = (scores_c >= th).astype(int)
            f1_c = f1_score(y_dev_c, preds_c, average="binary", zero_division=0)

            # Si el F1 mejora respecto al que tenía la clase ya, lo guardamos:
            if f1_c > max_f1_c:
                max_f1_c = f1_c
                best_th = th

        # En cada entrada 'i' de 'optimal_thresholds', almacenamos el mejor umbral encontrado para la clase 'c_i':
        optimal_thresholds[c] = best_th

    return optimal_thresholds

In [340]:
# Definimos asimismo una función para obtener las predicciones optimizadas dados los umbrales ajustados:
def predict_with_th(model, X, thresholds, verbose=False, alg="rl"):
    # Obtenemos las probabilidades originales:
    scores = model.predict_proba(X) if alg == "rl" else model.decision_function(X)

    # Actualizamos las predicciones con los umbrales de decisión que hemos calculado:
    preds = (scores >= thresholds).astype(int)

    # Buscamos los papers que se hayan quedado sin clase:
    zero_rows = np.where(preds.sum(axis=1) == 0)[0] # NumPy devuelve una tupla de 1 elemento, por eso lo extraemos

    # Iteramos sobre cada paper sin clase, obtenemos la clase de mayor probabilidad y se la asignamos:
    for r in zero_rows:
        top_class = np.argmax(scores[r])
        preds[r, top_class] = 1

    if verbose:
        # Mostramos los umbrales de decisión escogidos:
        for label, th in sorted(zip(mlb.classes_, thresholds), key=lambda x: x[1], reverse=True):
            print(f"{label} ({prob_c_dev[label]:.1%}): {th:.4f}", end="  ")
        print("\n")

        # Consultamos las métricas derivadas de usarlos:
        print("=== Métricas con umbrales de decisión optimizados ===")
        print(compute_metrics(y_dev, preds))

    return preds

In [ ]:
best_model_rl = rl[(best_c_rl, "balanced")]
best_th = adjust_thresholds(best_model_rl, B)
_ = predict_with_th(best_model_rl, B, best_th, verbose=True)

cs.IR (0.4%): 0.8557  math.OC (0.8%): 0.8458  cs.GR (0.9%): 0.6866  cs.CL (1.9%): 0.6617  cs.RO (3.3%): 0.6368  cs.CR (0.8%): 0.6219  cs.LG (52.3%): 0.5622  cs.CY (0.3%): 0.5125  eess.IV (3.9%): 0.5125  cs.NE (1.6%): 0.4926  cs.AI (13.9%): 0.4876  cs.DC (0.3%): 0.4776  eess.SP (0.6%): 0.4528  stat.ML (29.1%): 0.4378  cs.MA (0.7%): 0.4279  cs.SI (0.9%): 0.4279  cs.CV (61.3%): 0.3831  physics.chem-ph (0.0%): 0.3135  cs.SY (0.2%): 0.3085  eess.SY (0.2%): 0.3085  cs.MM (0.6%): 0.2388  stat.ME (0.1%): 0.1791  cs.IT (0.1%): 0.1742  math.IT (0.1%): 0.1742  math.ST (0.2%): 0.1344  stat.TH (0.2%): 0.1344  stat.AP (0.1%): 0.0896  cs.HC (0.1%): 0.0548  

=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.37310683585755217, 'f1': 0.35205075255183715, 'precision': 0.30766685024522783, 'recall': 0.5179573804448273, 'hamming_loss': 0.03529033389860242}


**Resultados:**
* Como podemos observar, la F1 macro ha aumentado notablemente de un **0.2990** a un **0.3521** tras ajustar los umbrales.

* Analizando la distribución de valores de los umbrales frente al tamaño de las clases, podemos apreciar que las categorías con mayores umbrales son las relativamente **medianas** (entre 0.4 % y 3.3 % de los *papers* las tienen como etiquetas), mientras que las clases más pequeñas (0.0 % - 0.1 % de las muestras las incluyen) cuentan con umbrales muy bajos.

    Esto se debe directamente al **balanceo de pesos** de las clases, que induce al clasificador a «arriegarse» a predecir clases poco frecuentes, lo cual para las medianas resulta en un número elevado de **falsos positivos**, mientras que para las que son realmente diminutas sigue sin representar un «impulso» suficiente para que el modelo las clasifique con confianza. De ahí que los umbrales de las primeras sean más estrictos y los de las más pequeñas sean muy flexibles. Para comprobar esta teoría, calcularemos los umbrales del mismo modelo sin balanceo de clases, donde, ahora sí, las clases de mayor tamaño recibirán los mayores umbrales: 

In [ ]:
best_model_rl_unb = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",
        C=best_c_rl,
        class_weight=None,
        random_state=SEED
    ),
    n_jobs=-1
).fit(A, y_train)

best_th_unb = adjust_thresholds(best_model_rl_unb, B)
_ = predict_with_th(best_model_rl_unb, B, best_th_unb, verbose=True)

cs.LG (52.3%): 0.5771  cs.CV (61.3%): 0.4826  stat.ML (29.1%): 0.2438  cs.AI (13.9%): 0.1841  math.OC (0.8%): 0.1742  cs.GR (0.9%): 0.1592  cs.RO (3.3%): 0.1592  cs.IR (0.4%): 0.1393  eess.IV (3.9%): 0.1344  cs.MA (0.7%): 0.1294  cs.CR (0.8%): 0.0995  cs.CL (1.9%): 0.0946  cs.SI (0.9%): 0.0896  cs.NE (1.6%): 0.0597  cs.MM (0.6%): 0.0548  cs.CY (0.3%): 0.0448  eess.SP (0.6%): 0.0349  cs.DC (0.3%): 0.0299  cs.SY (0.2%): 0.0299  eess.SY (0.2%): 0.0299  cs.HC (0.1%): 0.0100  cs.IT (0.1%): 0.0100  math.IT (0.1%): 0.0100  math.ST (0.2%): 0.0100  physics.chem-ph (0.0%): 0.0100  stat.AP (0.1%): 0.0100  stat.ME (0.1%): 0.0100  stat.TH (0.2%): 0.0100  

=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.4269340974212034, 'f1': 0.32269832548408267, 'precision': 0.32537101767587745, 'recall': 0.3599518091972792, 'hamming_loss': 0.03186217180281855}


* Como habíamos predicho en el párrafo anterior, con la salvedad de `math.OC`, los umbrales de decisión más altos se han asignado a las clases más frecuentes.

* Si bien el ajuste de umbrales ha conseguido subsanar en gran medida la indiferencia del modelo hacia las clases pequeñas, el clasificador ajustado sigue contando con una F1 macro ligeramente menor que la de aquel con balanceo de clases (0.3227 vs. 0.3521). Esto se debe a que el ajuste se realiza *después* de que el modelo calcule las probabilidades de cada categoría: como originalmente las probabilidades de las clases pequeñas se aplanaron en torno al cero, el ajuste de umbrales no ha podido recuperarlas completamente, de ahí que muchas de estas categorías cuenten ahora con el umbral mínimo que habíamos fijado (0.01).

### Parámetros del vectorizador

#### Uso de *stopwords*

Partiendo del modelo de RL que acabamos de optimizar, vamos a probar distintas variantes del clasificador. En primer lugar, comprobaremos si eliminar *stopwords* de la entrada influye en su rendimiento:

In [166]:
# Buscamos o descargamos el conjunto de stopwords en inglés:
try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')

In [ ]:
# Definimos un vectorizador que tenga en cuenta las stopwords:
vec_sw = TfidfVectorizer(token_pattern=TOKEN_PATTERN, stop_words=stop_words, sublinear_tf=True)
A_sw = vec_sw.fit_transform(train_tok)
B_sw = vec_sw.transform(dev_tok)

A_sw.sort_indices()
A_sw.sum_duplicates()
B_sw.sort_indices()

In [ ]:
# Entrenamos el modelo de RL con la nueva matriz dispersa:
best_model_rl_sw = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",
        C=best_c_rl,
        class_weight="balanced",
        random_state=SEED
    ),
    n_jobs=-1
).fit(A_sw, y_train)

# Comparamos nuestro mejor modelo (sin ajuste de umbrales) con y sin stopwords:
y_pred = best_model_rl.predict(B)
print("=== Con stopwords:")
print(compute_metrics(y_dev, y_pred))
print()

y_pred_sw = best_model_rl_sw.predict(B_sw)
print("=== Sin stopwords:")
print(compute_metrics(y_dev, y_pred_sw))

=== Con stopwords:
{'accuracy': 0.3874334834220221, 'f1': 0.29896037773550976, 'precision': 0.2641830308745444, 'recall': 0.36987166257123205, 'hamming_loss': 0.03380650254371089}

=== Sin stopwords:
{'accuracy': 0.38272615636512486, 'f1': 0.2930672043876307, 'precision': 0.25500120786107483, 'recall': 0.36743888115787293, 'hamming_loss': 0.03403309747967955}


**Resultados:**
* Podemos comprobar que la F1 macro para el modelo que elimina *stopwords* empeora ligeramente (0.2990 con ellas vs. 0.2931 sin ellas). Si bien esta diferencia es sutil en apariencia, representa una reducción de casi un **2 %** respecto de nuestro rendimiento original, por lo que las mantendremos.

#### Ajuste de `min_df`

A continuación, ajustaremos el parámetro `min_df` del vectorizador, que determina el **mínimo número de documentos** en los que debe aparecer un token para ser considerado dentro de la matriz TF-IDF. Hasta ahora habíamos mantenido su valor por defecto de 1, por lo que incluso los **hápax** quedaban incluidos en los datos:

In [ ]:
for min_df in [1, 2, 3]:
    # Definimos un vectorizador con la 'min_df' de la iteración actual:
    vec_mdf = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True, min_df=min_df)
    A_mdf = vec_mdf.fit_transform(train_tok)
    B_mdf = vec_mdf.transform(dev_tok)

    A_mdf.sort_indices()
    A_mdf.sum_duplicates()
    B_mdf.sort_indices()

    # Entrenamos el modelo de RL con cada 'min_df':
    best_model_rl_mdf = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",
            C=best_c_rl,
            class_weight="balanced",
            random_state=SEED
        ),
        n_jobs=-1
    ).fit(A_mdf, y_train)

    # Mostramos sus resultados:
    y_pred_mdf = best_model_rl_mdf.predict(B_mdf)
    print(f"=== min_df={min_df}:")
    print(compute_metrics(y_dev, y_pred_mdf))
    print()

=== min_df=1:
{'accuracy': 0.3874334834220221, 'f1': 0.29896037773550976, 'precision': 0.2641830308745444, 'recall': 0.36987166257123205, 'hamming_loss': 0.03380650254371089}

=== min_df=2:
{'accuracy': 0.3774048301268932, 'f1': 0.2959311244795361, 'precision': 0.2584497488644066, 'recall': 0.36910160522491686, 'hamming_loss': 0.03461054909069645}

=== min_df=3:
{'accuracy': 0.3708555055259926, 'f1': 0.2926060510398879, 'precision': 0.253002104297679, 'recall': 0.36931155338789445, 'hamming_loss': 0.03512221507514181}



**Resultados:**
* Como podemos observar, la diferencia entre las ejecuciones con cada `min_df` vuelve a ser pequeña pero tangible, prevaleciendo el valor original de 1 que teníamos fijado originalmente (0.2990 para `min_df=1` vs. ~0.2960 para el resto). El impacto es mayor si contemplamos la *accuracy*, cuya proporción de aciertos exactos cae más de 1.5 %. Con todo ello en mente, mantendremos nuestra configuración inicial.

#### N-gramas de palabras

Nuestro siguiente experimento consistirá en añadir bigramas y trigramas a la matriz TF-IDF en busca de mejoras en el rendimiento del clasificador —a costa de un aumento considerable de la cantidad de datos en memoria y el tiempo de entrenamiento—. En esta ocasión, dado el aumento del diccionario de términos a evaluar, deberemos ajustar el parámetro de regularización `C` a la par que probamos con los distintos tamaños de n-grama.

Concretamente, para los bigramas probaremos valores circundantes (pero en su mayoría menores) a nuestra mejor `C` actual, y para los trigramas haremos lo mismo, pero tomando como referencia la mejor `C` encontrada para el modelo de bigramas:

In [ ]:
# Definimos un vectorizador que considere unigramas y bigramas:
vec_bi = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), sublinear_tf=True)
A_bi = vec_bi.fit_transform(train_tok)
B_bi = vec_bi.transform(dev_tok)

A_bi.sort_indices()
A_bi.sum_duplicates()
B_bi.sort_indices()

In [ ]:
# Mejor modelo actual:
y_pred = best_model_rl.predict(B)
print("=== SOLO UNIGRAMAS:")
print(f"Rasgos: {A.shape[1]:_} palabras")
print()

print(f"== C={best_c_rl}:")
print(compute_metrics(y_dev, y_pred))
print()

# Modelo con unigramas y bigramas:
print(f"=== UNIGRAMAS Y BIGRAMAS:")
print(f"Rasgos: {A_bi.shape[1]:_} n-gramas")
print()

best_f1_bi = -1
best_c_bi = None
for c in [0.1, 0.5, 1, 2, best_c_rl, 4]:
    clf_bi = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",
            C=c,
            class_weight="balanced",
            random_state=SEED
        ),
        n_jobs=-1
    ).fit(A_bi, y_train)

    y_pred_bi = clf_bi.predict(B_bi)
    metrics_bi = compute_metrics(y_dev, y_pred_bi)

    if metrics_bi["f1"] > best_f1_bi:
        best_f1_bi = metrics_bi["f1"]
        best_c_bi = c

    print(f"== C={c}:")
    print(metrics_bi)
    print()

=== SOLO UNIGRAMAS:
Rasgos: 72_410 palabras

== C=2.82:
{'accuracy': 0.3874334834220221, 'f1': 0.29896037773550976, 'precision': 0.2641830308745444, 'recall': 0.36987166257123205, 'hamming_loss': 0.03380650254371089}

=== UNIGRAMAS Y BIGRAMAS:
Rasgos: 1_133_404 n-gramas

== C=0.1:
{'accuracy': 0.3004502660663119, 'f1': 0.26904085674894085, 'precision': 0.25188855117101744, 'recall': 0.38522790994587713, 'hamming_loss': 0.0413279925150576}

== C=0.5:
{'accuracy': 0.38436348751535, 'f1': 0.2802160499466707, 'precision': 0.3070414516115615, 'recall': 0.34718537277935846, 'hamming_loss': 0.03421583533126718}

== C=1:
{'accuracy': 0.42611543184609085, 'f1': 0.27366699377435627, 'precision': 0.311697595730443, 'recall': 0.3252609817850685, 'hamming_loss': 0.03134319630430969}

== C=2:
{'accuracy': 0.4578387228817028, 'f1': 0.26627773506394947, 'precision': 0.2546131511608649, 'recall': 0.2997840801673407, 'hamming_loss': 0.029252675282147242}

== C=2.82:
{'accuracy': 0.46970937372083504, 'f1

In [ ]:
# Definimos otro vectorizador que incluya trigramas aparte de los bigramas y los unigramas:
vec_tri = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 3), sublinear_tf=True)
A_tri = vec_tri.fit_transform(train_tok)
B_tri = vec_tri.transform(dev_tok)

A_tri.sort_indices()
A_tri.sum_duplicates()
B_tri.sort_indices()

In [ ]:
print(f"=== UNIGRAMAS, BIGRAMAS Y TRIGRAMAS:")
print(f"Rasgos: {A_tri.shape[1]:_} n-gramas")
print()

best_f1_tri = -1
best_c_tri = None
for c in [0.05, 0.1, 0.25, best_c_bi, 0.75, 1]:
    clf_tri = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",
            C=c,
            class_weight="balanced",
            random_state=SEED
        ),
        n_jobs=-1
    ).fit(A_tri, y_train)

    y_pred_tri = clf_tri.predict(B_tri)
    metrics_tri = compute_metrics(y_dev, y_pred_tri)

    if (metrics_tri["f1"] > best_f1_tri):
        best_f1_tri = metrics_tri["f1"]
        best_c_tri = c

    print(f"== C={c}:")
    print(metrics_tri)
    print()

=== UNIGRAMAS, BIGRAMAS Y TRIGRAMAS:
Rasgos: 4_020_415 n-gramas

== C=0.05:
{'accuracy': 0.3192795742939009, 'f1': 0.23650370327993292, 'precision': 0.21264804474326304, 'recall': 0.32989786692886464, 'hamming_loss': 0.039749137477340504}

== C=0.1:
{'accuracy': 0.3362668849774867, 'f1': 0.24001983994654771, 'precision': 0.22105305899290473, 'recall': 0.32398420967628977, 'hamming_loss': 0.0381483538974329}

== C=0.25:
{'accuracy': 0.37249283667621774, 'f1': 0.24486669586619514, 'precision': 0.22929658119907984, 'recall': 0.314850686662656, 'hamming_loss': 0.03534881001111046}

== C=0.5:
{'accuracy': 0.40830945558739257, 'f1': 0.2503703678201262, 'precision': 0.24036282234140147, 'recall': 0.3070924357521401, 'hamming_loss': 0.03274662300450266}

== C=0.75:
{'accuracy': 0.42570609905853457, 'f1': 0.25329237610831057, 'precision': 0.24713896823580356, 'recall': 0.3013049967077382, 'hamming_loss': 0.03151862464183381}

== C=1:
{'accuracy': 0.4386000818665575, 'f1': 0.24794516853231277, '

**Resultados:**
* Este experimento arroja una conclusión bastante clara: **no debemos trabajar con n-gramas**. Mientras que nuestro diccionario de términos original solo contaba con ~72 000 palabras, añadirle bigramas y trigramas supone ampliarlo a **>1.1 M y >4 M** de rasgos, respectivamente, solo para acabar empeorando las métricas de evaluación en ambos casos.

#### N-gramas de caracteres

Como último experimento de esta sección nos mantendremos en la línea del anterior, en esta ocasión trabajando con n-gramas de **caracteres**. Empezaremos tomando n-gramas de tamaños entre 2 y 4, y ajustaremos el parámetro `C` con valores entre 0.5 y 50:

In [293]:
# Creamos una función que reciba el tamaño de los n-gramas y nos defina el vectorizador:
def vectorize_chgrams(ngram_range):
    vec_ch = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram_range, sublinear_tf=True, min_df=1)
    A_ch = vec_ch.fit_transform(train_tok)
    B_ch = vec_ch.transform(dev_tok)

    A_ch.sort_indices()
    A_ch.sum_duplicates()
    B_ch.sort_indices()

    print(f"Rasgos: {A_ch.shape[1]:_} n-gramas de caracteres frente a {A.shape[1]:_} con palabras")
    print()

    return A_ch, B_ch

In [294]:
# Envolvemos dentro de una función el entrenamiento del modelo:
def train_chgrams(c, A_ch, B_ch, ch_size, alg="rl"):
    # Definimos el modelo:
    if alg == "rl":
        clf_ch = OneVsRestClassifier(
            LogisticRegression(
                solver="liblinear",
                C=c,
                class_weight="balanced",
                random_state=SEED
            ),
            n_jobs=12
        ).fit(A_ch, y_train)
    elif alg == "svm":
        clf_ch = OneVsRestClassifier(
            LinearSVC(
                C=c,
                dual=False,
                class_weight="balanced",
                max_iter=8000,
                tol=1e-3,
                random_state=SEED,
            ),
            n_jobs=12,
        ).fit(A_ch, y_train)  

    # Realizamos la predicción y obtenemos las métricas:
    y_pred_ch = clf_ch.predict(B_ch)
    metrics_ch = compute_metrics(y_dev, y_pred_ch)

    # Guardamos el F1 actual en el historial:
    current_f1 = metrics_ch["f1"]
    history[ch_size][c] = current_f1

    # Comprobamos si es el mejor F1 hasta ahora.
    # Si lo es, actualizamos las variables correspondientes:
    if current_f1 > best[ch_size]["f1"]:
        best[ch_size]["f1"] = current_f1
        best[ch_size]["model"] = clf_ch
        best[ch_size]["c"] = c

    # Imprimimos las métricas:
    print(f"== C={c}:")
    print(metrics_ch)
    print()

In [316]:
history, best = {}, {}

In [317]:
ch_size = (2, 4)
A_ch, B_ch = vectorize_chgrams(ch_size)
history[ch_size] = {}
best[ch_size] = {"f1": -1, "c": None, "model": None}

for c in [0.5, 1, 2, best_c_rl, 5, 10, 20, 50]:
    train_chgrams(c, A_ch, B_ch, ch_size, alg="rl")

Rasgos: 92_189 n-gramas de caracteres frente a 72_410 con palabras

== C=0.5:
{'accuracy': 0.23986901350798198, 'f1': 0.27933797286943085, 'precision': 0.2120999338923384, 'recall': 0.5539093805400017, 'hamming_loss': 0.04798695982691071}

== C=1:
{'accuracy': 0.2816209578387229, 'f1': 0.3017746143512385, 'precision': 0.23271499534820242, 'recall': 0.5354297868484301, 'hamming_loss': 0.04330887082626747}

== C=2:
{'accuracy': 0.3215309046254605, 'f1': 0.31618714858806213, 'precision': 0.25316242856452276, 'recall': 0.4975172018990404, 'hamming_loss': 0.03947868545699082}

== C=2.82:
{'accuracy': 0.3417928776094965, 'f1': 0.31875028107070263, 'precision': 0.2610026847403192, 'recall': 0.47563298886920435, 'hamming_loss': 0.03783404479270218}

== C=5:
{'accuracy': 0.3692181743757675, 'f1': 0.3232154758161108, 'precision': 0.28081966021006777, 'recall': 0.4391805359505608, 'hamming_loss': 0.035685047658031696}

== C=10:
{'accuracy': 0.3902988129349161, 'f1': 0.3237596589633377, 'precision

**Resultados:**
* Los hallazgos de estas pruebas resultan bastante prometedores: hemos encontrado un pico en `C=20` con una F1 macro de **0.3320** sin haber ajustado umbrales. En contrapartida, si bien solo contamos con 92 mil n-gramas de caracteres (bastantes menos que al tratar con sus análogos de palabras), la matriz TF-IDF se ha vuelto mucho más densa a raíz de una mayor repetición de cada rasgo, lo que deriva en un aumento severo del tiempo de entrenamiento.

* A continuación, vamos a probar otros tamaños de n-grama para esa `C=20` y comprobar si suponen una mejora de rendimiento:

In [318]:
for ch_size in [(2, 3), (2, 5), (3, 4), (3, 5)]:
    print(f"== Tamaños de n-grama: {ch_size}")
    A_ch, B_ch = vectorize_chgrams(ch_size)
    
    history[ch_size] = {}
    best[ch_size] = {"f1": -1, "c": None, "model": None}

    train_chgrams(20, A_ch, B_ch, ch_size, alg="rl")

== Tamaños de n-grama: (2, 3)
Rasgos: 19_208 n-gramas de caracteres frente a 72_410 con palabras

== C=20:
{'accuracy': 0.36062218583708555, 'f1': 0.3054304605957126, 'precision': 0.2787751417414922, 'recall': 0.378194066296666, 'hamming_loss': 0.03657680837377931}

== Tamaños de n-grama: (2, 5)
Rasgos: 227_131 n-gramas de caracteres frente a 72_410 con palabras

== C=20:
{'accuracy': 0.4134261154318461, 'f1': 0.33138452102308874, 'precision': 0.33913977561522407, 'recall': 0.3585933029238008, 'hamming_loss': 0.03206683819659669}

== Tamaños de n-grama: (3, 4)
Rasgos: 90_749 n-gramas de caracteres frente a 72_410 con palabras

== C=20:
{'accuracy': 0.4054441260744986, 'f1': 0.3365585695050723, 'precision': 0.3372715861480899, 'recall': 0.3744443230510231, 'hamming_loss': 0.032841646687328224}

== Tamaños de n-grama: (3, 5)
Rasgos: 225_691 n-gramas de caracteres frente a 72_410 con palabras

== C=20:
{'accuracy': 0.41567744576340565, 'f1': 0.3295557787962579, 'precision': 0.338549293641

* Emplear los tamaños (3, 4) de n-grama nos reporta una F1 de **0.3366**, por lo que a partir de ahora trabajaremos con ellos. 

* Vamos a intentar acotar un poco más el valor de `C` dentro del rango [10, 50] mediante una **búsqueda ternaria**. A tal fin, definiremos una función que evaluará en cada iteración los dos puntos medios entre la mejor `C` actual y sus dos vecinas contiguas más cercanas en el historial de resultados:

In [315]:
# Definimos la función de búsqueda ternaria:
def ternary_search(A_ch, B_ch, ch_size, alg="rl", n_iter=6):
    for _ in range(n_iter):
        # Obtenemos la lista de C's evaluadas y el índice en ella de la mejor C:
        evaluated_c = sorted(history[ch_size].keys())
        idx = evaluated_c.index(best[ch_size]["c"])

        # Creamos una lista con los puntos medios a evaluar:
        midpoints = []

        # Si la mejor C no es la primera en la lista,
        # obtenemos su vecina izquierda y añadimos el punto medio
        # entre ella y la mejor C a la lista de puntos medios:
        if idx > 0:
            c_left = evaluated_c[idx - 1]
            midpoints.append((c_left + best[ch_size]["c"]) / 2)

        # Hacemos lo mismo con la vecina derecha si C no es la última de la lista:
        if idx < len(evaluated_c) - 1:
            c_right = evaluated_c[idx + 1]
            midpoints.append((best[ch_size]["c"] + c_right) / 2)

        # Evaluamos cada punto medio:
        for mid in midpoints:
            train_chgrams(mid, A_ch, B_ch, ch_size, alg=alg)

In [325]:
best_ch_size = (3, 4)
A_ch, B_ch = vectorize_chgrams(best_ch_size)

# Entrenamos el modelo para C=10 y C=50 a fin de tener vecinos en el historial:
for c in [10, 50]:
    train_chgrams(c, A_ch, B_ch, best_ch_size, alg="rl")

# Ejecutamos la búsqueda ternaria:
ternary_search(A_ch, B_ch, best_ch_size, alg="rl")

Rasgos: 90_749 n-gramas de caracteres frente a 72_410 con palabras

== C=10:
{'accuracy': 0.3948014735980352, 'f1': 0.3220973723164242, 'precision': 0.30237606425740393, 'recall': 0.39219928139455257, 'hamming_loss': 0.03379919302964739}

== C=50:
{'accuracy': 0.40749079001228, 'f1': 0.3207381683598048, 'precision': 0.33691995946342573, 'recall': 0.3354101865497673, 'hamming_loss': 0.0326515993216771}

== C=15.0:
{'accuracy': 0.401760130986492, 'f1': 0.33235105008534316, 'precision': 0.31906106591228794, 'recall': 0.3809466458436407, 'hamming_loss': 0.03319250336237647}

== C=35.0:
{'accuracy': 0.40626279164961115, 'f1': 0.32698301997448687, 'precision': 0.3395333092208824, 'recall': 0.3468211752242884, 'hamming_loss': 0.03271007543418514}

== C=17.5:
{'accuracy': 0.4044207941056079, 'f1': 0.33367277319331407, 'precision': 0.32225375337578066, 'recall': 0.37816990480948415, 'hamming_loss': 0.03306093210923338}

== C=27.5:
{'accuracy': 0.40667212443716744, 'f1': 0.32766838449257385, 'pr

* La búsqueda ternaria ha tenido éxito y hemos conseguido mejorar la F1 hasta un **0.3372** para `C=20.46875`. Ya solo nos queda comprobar si este valor mejora ajustando los umbrales de decisión del clasificador:

In [323]:
# Guardamos en 'best_model_rl' el mejor modelo de n-gramas que hemos conseguido:
best_model_rl = best[best_ch_size]["model"]

# Ajustamos sus umbrales de decisión:
char_th = adjust_thresholds(best_model_rl, B_ch)
_ = predict_with_th(best_model_rl, B_ch, char_th, verbose=True)

cs.CL (1.9%): 0.8010  cs.GR (0.9%): 0.6766  math.ST (0.2%): 0.6120  stat.TH (0.2%): 0.6120  cs.SI (0.9%): 0.5871  cs.RO (3.3%): 0.5771  physics.chem-ph (0.0%): 0.5722  cs.LG (52.3%): 0.5324  cs.AI (13.9%): 0.5025  eess.SP (0.6%): 0.4926  cs.SY (0.2%): 0.4876  eess.SY (0.2%): 0.4876  cs.CR (0.8%): 0.4677  cs.MA (0.7%): 0.4478  math.OC (0.8%): 0.4428  cs.IR (0.4%): 0.4080  stat.ML (29.1%): 0.3980  cs.CV (61.3%): 0.3732  eess.IV (3.9%): 0.3632  cs.NE (1.6%): 0.3483  cs.IT (0.1%): 0.2737  math.IT (0.1%): 0.2737  stat.AP (0.1%): 0.2587  cs.CY (0.3%): 0.2438  cs.MM (0.6%): 0.1443  cs.DC (0.3%): 0.1095  cs.HC (0.1%): 0.0349  stat.ME (0.1%): 0.0299  

=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.37822349570200575, 'f1': 0.37861445657790255, 'precision': 0.37476529962230654, 'recall': 0.4544247651373097, 'hamming_loss': 0.035253786328284895}


* Efectivamente, hemos obtenido una F1 macro de **0.3786** para esta versión final del clasificador RL, razonablemente superior al 0.3366 con el que contábamos antes del ajuste de umbrales y el 0.3521 del modelo sin n-gramas de caracteres.

### Resultados del test para RL - [BORRAR AL ENTREGAR]

In [ ]:
raw_test, y_test_nobin = load_corpus(TEST_FILE)
print("Papers de test:", len(raw_test))

y_test = mlb.transform(y_test_nobin)

test_tok = [tokenize(title) + " " + tokenize(summary) for title, summary in raw_test]

Papers de test: 4885


In [245]:
labels_train = [cat for paper_cats in mlb.inverse_transform(y_train) for cat in paper_cats]
labels_dev = [cat for paper_cats in mlb.inverse_transform(y_dev) for cat in paper_cats]
labels_test = [cat for paper_cats in y_test_nobin for cat in paper_cats]

prob_c_train = show_stats(train_tok, labels_train, "Entrenamiento")
print("Clases distintas:", len(set(labels_train)))
print()
prob_c_dev = show_stats(dev_tok, labels_dev, "Validación")
print("Clases distintas:", len(set(labels_dev)))
print()
prob_c_test = show_stats(test_tok, labels_test, "Test")
print("Clases distintas:", len(set(labels_test)))
print()

Entrenamiento
-------------
Muestras: 27684   Tokens: 4836492   Tokens por muestra: 174.70
cs.CV: 16786 (60.6%)  cs.LG: 14608 (52.8%)  stat.ML: 8142 (29.4%)  cs.AI: 3819 (13.8%)  eess.IV: 1071 (3.9%)  cs.RO: 894 (3.2%)  cs.CL: 609 (2.2%)  cs.NE: 431 (1.6%)  cs.CR: 235 (0.8%)  cs.SI: 219 (0.8%)  math.OC: 219 (0.8%)  cs.GR: 215 (0.8%)  eess.SP: 198 (0.7%)  cs.MM: 178 (0.6%)  cs.MA: 145 (0.5%)  cs.IR: 117 (0.4%)  cs.DC: 72 (0.3%)  cs.CY: 67 (0.2%)  cs.SY: 57 (0.2%)  eess.SY: 57 (0.2%)  stat.ME: 42 (0.2%)  stat.AP: 40 (0.1%)  cs.IT: 37 (0.1%)  math.IT: 37 (0.1%)  math.ST: 35 (0.1%)  stat.TH: 35 (0.1%)  cs.HC: 35 (0.1%)  physics.chem-ph: 19 (0.1%)  

Clases distintas: 28

Validación
----------
Muestras: 4886   Tokens: 856911   Tokens por muestra: 175.38
cs.CV: 2996 (61.3%)  cs.LG: 2554 (52.3%)  stat.ML: 1422 (29.1%)  cs.AI: 677 (13.9%)  eess.IV: 190 (3.9%)  cs.RO: 161 (3.3%)  cs.CL: 95 (1.9%)  cs.NE: 78 (1.6%)  cs.SI: 44 (0.9%)  cs.GR: 44 (0.9%)  math.OC: 40 (0.8%)  cs.CR: 39 (0.8%)  cs.MA:

In [ ]:
vec_test = TfidfVectorizer(analyzer="char_wb", ngram_range=best_ch_size, sublinear_tf=True, min_df=1)
vec_test.fit_transform(train_tok)
X_test = vec_test.transform(test_tok)
X_test.sort_indices()

preds_test = predict_with_th(best_model_rl, X_test, char_th, verbose=False)
print(compute_metrics(y_test, preds_test))

{'accuracy': 0.25670419651995907, 'f1': 0.16639097501679462, 'precision': 0.1520282972397664, 'recall': 0.24462867788139178, 'hamming_loss': 0.04346395671881854}


## Máquina de Vectores de Soporte (SVM)

En esta sección repetiremos los experimentos realizados sobre la regresión logística para un clasificador basado en SVM (en su implementación `LinearSVC` de Scikit). Reduciremos el número de comentarios a fin de evitar redundancia.

In [248]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

A.sort_indices()
A.sum_duplicates()
B.sort_indices()

### Parámetros del clasificador

Ajustamos del parámetro `C` con balanceo de clases:

In [ ]:
svm = {}
for c in [0.01, 0.05, 0.1, 0.25, 0.5, 1, 2, 5]:

    svm[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    ).fit(A, y_train)

    y_pred = svm[c].predict(B)
    print(f"=== C={c}:")
    print(compute_metrics(y_dev, y_pred))
    print()

=== C=0.01:
{'accuracy': 0.20384772820302907, 'f1': 0.25831645889706584, 'precision': 0.1978161189731125, 'recall': 0.5081925303480189, 'hamming_loss': 0.05191216887901293}

=== C=0.05:
{'accuracy': 0.30597625869832173, 'f1': 0.2918011128596927, 'precision': 0.23790905862304154, 'recall': 0.4347488377456982, 'hamming_loss': 0.040567803052453076}

=== C=0.1:
{'accuracy': 0.3534588620548506, 'f1': 0.2948080662836616, 'precision': 0.269611882191975, 'recall': 0.386105081979478, 'hamming_loss': 0.03651833226127127}

=== C=0.25:
{'accuracy': 0.40523945968072045, 'f1': 0.2805201628985206, 'precision': 0.31712293105958034, 'recall': 0.3241136566692634, 'hamming_loss': 0.032417694871644934}

=== C=0.5:
{'accuracy': 0.4269340974212034, 'f1': 0.2719207502800166, 'precision': 0.3212972687843381, 'recall': 0.287163857383482, 'hamming_loss': 0.030875387404245367}

=== C=1:
{'accuracy': 0.4394187474416701, 'f1': 0.263357645444275, 'precision': 0.3374184406793509, 'recall': 0.25534762817020457, 'hamm

Afinamos la búsqueda en el rango [0.05; 0.25]:

In [ ]:
max_f1_svm = -1
for c in np.arange(0.06, 0.25, 0.01):
    c = round(c, 2)

    svm[c] = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    ).fit(A, y_train)
    
    y_pred = svm[c].predict(B)
    f1_svm = compute_metrics(y_dev, y_pred)['f1']
    if f1_svm > max_f1_svm:
        max_f1_svm = f1_svm
        best_c_svm = c

print("Mejor C:", best_c_svm)
print("Mejor F1 macro:", max_f1_svm)

Mejor C: 0.09
Mejor F1 macro: 0.29790226676852694


**Resultados:**
* La mejor `C` que hemos obtenido es de **0.09**, la cual nos ha proporcionado una F1 macro de **0.2979**, prácticamente idéntica a aquel 0.2978 que obtuvimos en las mismas condiciones para la RL.

* El motivo detrás de que las `C` que funcionan bien en SVM sean bastante más bajas que en RL (indicando que el modelo necesita mayor regularización) se debe a las funciones de pérdida que albergan ambos mecanismos. En términos simples, mientras que todas las muestras aportan a la pérdida en RL, en SVM solo contribuyen las más **ambiguas**. Es decir, aquellas que se acercan mucho a la frontera de decisión. Si no imponemos cierto nivel de regularización al modelo, intentará por todos los medios dibujar una frontera retorcida y enrevesada alrededor de esas muestras ambiguas, la cual luego no se traducirá correctamente al conjunto de validación.

### Parámetros del vectorizador

#### Uso de *stopwords*

Volvemos a definir un vectorizador TF-IDF que excluya el conjunto de *stopwords* del diccionario de términos y entrenamos nuestro mejor modelo de SVM con él:

In [252]:
vec_sw = TfidfVectorizer(token_pattern=TOKEN_PATTERN, stop_words=stop_words, sublinear_tf=True)
A_sw = vec_sw.fit_transform(train_tok)
B_sw = vec_sw.transform(dev_tok)

A_sw.sort_indices()
A_sw.sum_duplicates()
B_sw.sort_indices()

In [253]:
# Entrenamos el modelo de RL con la nueva matriz dispersa:
best_model_svm_sw = OneVsRestClassifier(
        LinearSVC(
            C=best_c_svm,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    ).fit(A_sw, y_train)

# Comparamos nuestro mejor modelo con y sin stopwords:
best_model_svm = svm[best_c_svm]
y_pred = best_model_svm.predict(B)
print("=== Con stopwords:")
print(compute_metrics(y_dev, y_pred))
print()

y_pred_sw = best_model_svm_sw.predict(B_sw)
print("=== Sin stopwords:")
print(compute_metrics(y_dev, y_pred_sw))

=== Con stopwords:
{'accuracy': 0.3448628735161686, 'f1': 0.29790226676852694, 'precision': 0.27061395436428537, 'recall': 0.39558192558276495, 'hamming_loss': 0.03715425998479621}

=== Sin stopwords:
{'accuracy': 0.34650020466639375, 'f1': 0.3011955029177487, 'precision': 0.27299164719333185, 'recall': 0.39637384573549167, 'hamming_loss': 0.037015379217589614}


**Resultados:**

* En esta ocasión, sí que vamos a eliminar las *stopwords* del vocabulario, teniendo en cuenta que su exclusión ha supuesto un aumento de la F1 macro de 0.2979 a **0.3011**:

In [255]:
best_model_svm = best_model_svm_sw

#### Ajuste de `min_df`

In [254]:
for min_df in [1, 2, 3]:
    # Definimos un vectorizador con la 'min_df' de la iteración actual:
    vec_mdf = TfidfVectorizer(token_pattern=TOKEN_PATTERN, stop_words=stop_words, sublinear_tf=True, min_df=min_df)
    A_mdf = vec_mdf.fit_transform(train_tok)
    B_mdf = vec_mdf.transform(dev_tok)

    A_mdf.sort_indices()
    A_mdf.sum_duplicates()
    B_mdf.sort_indices()

    # Entrenamos el modelo de SVM con cada 'min_df':
    best_model_svm_mdf = OneVsRestClassifier(
        LinearSVC(
            C=best_c_svm,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    ).fit(A_mdf, y_train)

    # Mostramos sus resultados:
    y_pred_mdf = best_model_svm_mdf.predict(B_mdf)
    print(f"=== min_df={min_df}:")
    print(compute_metrics(y_dev, y_pred_mdf))
    print()

=== min_df=1:
{'accuracy': 0.34650020466639375, 'f1': 0.3011955029177487, 'precision': 0.27299164719333185, 'recall': 0.39637384573549167, 'hamming_loss': 0.037015379217589614}

=== min_df=2:
{'accuracy': 0.33462955382726156, 'f1': 0.29231376774898277, 'precision': 0.26015866830974615, 'recall': 0.3917570283031858, 'hamming_loss': 0.03783404479270218}

=== min_df=3:
{'accuracy': 0.3348342202210397, 'f1': 0.28895345307888337, 'precision': 0.24764754576895048, 'recall': 0.39397622288584827, 'hamming_loss': 0.038097187298988366}



**Resultados:**

* Igual que en la regresión logística, vamos a mantener el valor por defecto de **1** para `min_df`.

#### N-gramas de palabras

In [ ]:
vec_bi = TfidfVectorizer(token_pattern=TOKEN_PATTERN, stop_words=stop_words, ngram_range=(1, 2), sublinear_tf=True)
A_bi = vec_bi.fit_transform(train_tok)
B_bi = vec_bi.transform(dev_tok)

A_bi.sort_indices()
A_bi.sum_duplicates()
B_bi.sort_indices()

In [ ]:
# Mejor modelo actual:
y_pred = best_model_svm.predict(B_sw)
print("=== SOLO UNIGRAMAS:")
print(f"Rasgos: {A_sw.shape[1]:_} palabras")
print()

print(f"== C={best_c_svm}:")
print(compute_metrics(y_dev, y_pred))
print()

# Modelo con unigramas y bigramas:
print(f"=== UNIGRAMAS Y BIGRAMAS:")
print(f"Rasgos: {A_bi.shape[1]:_} n-gramas")
print()

best_f1_bi = -1
best_c_bi = None
for c in [0.01, 0.05, best_c_svm, 0.1, 0.2, 0.5]:
    clf_bi = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    ).fit(A_bi, y_train)

    y_pred_bi = clf_bi.predict(B_bi)
    metrics_bi = compute_metrics(y_dev, y_pred_bi)

    if metrics_bi["f1"] > best_f1_bi:
        best_f1_bi = metrics_bi["f1"]
        best_c_bi = c

    print(f"== C={c}:")
    print(metrics_bi)
    print()

=== SOLO UNIGRAMAS:
Rasgos: 72_275 palabras

== C=0.09:
{'accuracy': 0.34650020466639375, 'f1': 0.3011955029177487, 'precision': 0.27299164719333185, 'recall': 0.39637384573549167, 'hamming_loss': 0.037015379217589614}

=== UNIGRAMAS Y BIGRAMAS:
Rasgos: 1_458_608 n-gramas

== C=0.01:
{'accuracy': 0.27363896848137537, 'f1': 0.2658295306648254, 'precision': 0.2460983700660355, 'recall': 0.39669543349027203, 'hamming_loss': 0.043645108473188705}

== C=0.05:
{'accuracy': 0.3825214899713467, 'f1': 0.2619889689236144, 'precision': 0.22979707036580463, 'recall': 0.3437300771915571, 'hamming_loss': 0.03472750131571253}

== C=0.09:
{'accuracy': 0.4279574293900941, 'f1': 0.2684781946528365, 'precision': 0.2507984236347875, 'recall': 0.32274018606992716, 'hamming_loss': 0.03144552950119876}

== C=0.1:
{'accuracy': 0.4369627507163324, 'f1': 0.2685713870533997, 'precision': 0.2540920917208094, 'recall': 0.3197278903758759, 'hamming_loss': 0.030890006432372375}

== C=0.2:
{'accuracy': 0.472370036839

In [ ]:
vec_tri = TfidfVectorizer(token_pattern=TOKEN_PATTERN, stop_words=stop_words, ngram_range=(1, 3), sublinear_tf=True)
A_tri = vec_tri.fit_transform(train_tok)
B_tri = vec_tri.transform(dev_tok)

A_tri.sort_indices()
A_tri.sum_duplicates()
B_tri.sort_indices()

In [260]:
print(f"=== UNIGRAMAS, BIGRAMAS Y TRIGRAMAS:")
print(f"Rasgos: {A_tri.shape[1]:_} n-gramas")
print()

best_f1_tri = -1
best_c_tri = None
for c in [0.01, 0.05, 0.07, best_c_svm, best_c_bi, 0.2, 0.5]:
    clf_tri = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    ).fit(A_tri, y_train)

    y_pred_tri = clf_tri.predict(B_tri)
    metrics_tri = compute_metrics(y_dev, y_pred_tri)

    if (metrics_tri["f1"] > best_f1_tri):
        best_f1_tri = metrics_tri["f1"]
        best_c_tri = c

    print(f"== C={c}:")
    print(metrics_tri)
    print()

=== UNIGRAMAS, BIGRAMAS Y TRIGRAMAS:
Rasgos: 4_114_656 n-gramas

== C=0.01:
{'accuracy': 0.286123618501842, 'f1': 0.24026764274264695, 'precision': 0.2048089065324051, 'recall': 0.3627465349432927, 'hamming_loss': 0.042914157066838196}

== C=0.05:
{'accuracy': 0.37065083913221447, 'f1': 0.25161145409467073, 'precision': 0.22527893532992388, 'recall': 0.33495233633543525, 'hamming_loss': 0.035743523770539735}

== C=0.07:
{'accuracy': 0.39807613589848545, 'f1': 0.2534479234316805, 'precision': 0.23135279799044234, 'recall': 0.32389375756606914, 'hamming_loss': 0.03374802643120285}

== C=0.09:
{'accuracy': 0.41363078182562424, 'f1': 0.2566778474006165, 'precision': 0.23865310111619584, 'recall': 0.3162434105840904, 'hamming_loss': 0.032505409040406996}

== C=0.1:
{'accuracy': 0.4197707736389685, 'f1': 0.25594680860688246, 'precision': 0.23932847355385026, 'recall': 0.31154795379882216, 'hamming_loss': 0.03204490965440618}

== C=0.2:
{'accuracy': 0.46275071633237824, 'f1': 0.25694862758601

**Resultados:**

* En vista del claro empeoramiento de las métricas de evaluación en ambos experimentos, nuevamente seguiremos el ejemplo de la regresión logística y no consideraremos n-gramas de palabras para entrenar nuestro clasificador.

#### N-gramas de caracteres

In [333]:
history, best = {}, {}

In [334]:
ch_size = (2, 4)
A_ch, B_ch = vectorize_chgrams(ch_size)
history[ch_size] = {}
best[ch_size] = {"f1": -1, "c": None, "model": None}

for c in [0.05, best_c_svm, 0.1, 0.2, 0.5, 1, 2]:
    train_chgrams(c, A_ch, B_ch, ch_size, alg="svm")

Rasgos: 92_189 n-gramas de caracteres frente a 72_410 con palabras

== C=0.05:
{'accuracy': 0.24887433483422022, 'f1': 0.2943662368060521, 'precision': 0.22857370912680205, 'recall': 0.5379440520316757, 'hamming_loss': 0.04656891409859073}

== C=0.09:
{'accuracy': 0.2857142857142857, 'f1': 0.3027015746873974, 'precision': 0.24483249585086525, 'recall': 0.4956273478604309, 'hamming_loss': 0.04240249108239284}

== C=0.1:
{'accuracy': 0.29144494474007365, 'f1': 0.305671062544416, 'precision': 0.2482033200410314, 'recall': 0.49159171110005534, 'hamming_loss': 0.041678849190105845}

== C=0.2:
{'accuracy': 0.3379042161277118, 'f1': 0.32721203776417157, 'precision': 0.28997988304519834, 'recall': 0.44968823774938377, 'hamming_loss': 0.03779749722238466}

== C=0.5:
{'accuracy': 0.38313548915268114, 'f1': 0.2890439145245021, 'precision': 0.2881475483910644, 'recall': 0.33656652092578027, 'hamming_loss': 0.03420852581720367}

== C=1:
{'accuracy': 0.40769545640605814, 'f1': 0.28068493031318337, '

Parece que usar n-gramas de caracteres nos va a ayudar en SVM también. Probemos distintos tamaños de n-grama para `C=0.2`:

In [335]:
for ch_size in [(2, 3), (2, 5), (3, 4), (3, 5)]:
    A_ch, B_ch = vectorize_chgrams(ch_size)
    history[ch_size] = {}
    best[ch_size] = {"f1": -1, "c": None, "model": None}

    print(f"== Tamaños de n-grama: {ch_size}")
    train_chgrams(0.2, A_ch, B_ch, ch_size, alg="svm")

Rasgos: 19_208 n-gramas de caracteres frente a 72_410 con palabras

== Tamaños de n-grama: (2, 3)
== C=0.2:
{'accuracy': 0.28489562013917313, 'f1': 0.31181985701709586, 'precision': 0.26181735832144254, 'recall': 0.47124644274313443, 'hamming_loss': 0.042753347757441086}

Rasgos: 227_131 n-gramas de caracteres frente a 72_410 con palabras

== Tamaños de n-grama: (2, 5)
== C=0.2:
{'accuracy': 0.3548915268112976, 'f1': 0.3201331276882051, 'precision': 0.3033147742655613, 'recall': 0.4258056750785998, 'hamming_loss': 0.03632097538155663}

Rasgos: 90_749 n-gramas de caracteres frente a 72_410 con palabras

== Tamaños de n-grama: (3, 4)
== C=0.2:
{'accuracy': 0.34302087597216535, 'f1': 0.33074404986459266, 'precision': 0.30698034441299304, 'recall': 0.4429748006975864, 'hamming_loss': 0.0372858312379393}

Rasgos: 225_691 n-gramas de caracteres frente a 72_410 con palabras

== Tamaños de n-grama: (3, 5)
== C=0.2:
{'accuracy': 0.35611952517396644, 'f1': 0.32288899422133893, 'precision': 0.307

In [338]:
best_ch_size = (3, 4)
A_ch, B_ch = vectorize_chgrams(best_ch_size)

# Entrenamos el modelo para C=0.1 y C=0.5 a fin de tener vecinos en el historial:
for c in [0.1, 0.5]:
    train_chgrams(c, A_ch, B_ch, best_ch_size, alg="svm")

# Ejecutamos la búsqueda ternaria:
ternary_search(A_ch, B_ch, best_ch_size, alg="svm")

Rasgos: 90_749 n-gramas de caracteres frente a 72_410 con palabras

== C=0.1:
{'accuracy': 0.29758493655341794, 'f1': 0.3104522301203218, 'precision': 0.2574187065965634, 'recall': 0.4887501557320224, 'hamming_loss': 0.04120373077597801}

== C=0.5:
{'accuracy': 0.38456815390912813, 'f1': 0.2898148170217368, 'precision': 0.2901532998602794, 'recall': 0.3344575456104019, 'hamming_loss': 0.03403309747967955}

== C=0.15000000000000002:
{'accuracy': 0.32460090053213264, 'f1': 0.32586924630531405, 'precision': 0.28605045562122716, 'recall': 0.470655179834074, 'hamming_loss': 0.038864686275656396}

== C=0.35:
{'accuracy': 0.37146950470732704, 'f1': 0.3355996998384493, 'precision': 0.3299317578106363, 'recall': 0.4006623910572634, 'hamming_loss': 0.03507104847669727}

== C=0.275:
{'accuracy': 0.36021285304952927, 'f1': 0.3366336778254116, 'precision': 0.32404642562049407, 'recall': 0.41841748340685486, 'hamming_loss': 0.0359774282205719}

== C=0.425:
{'accuracy': 0.3780188293082276, 'f1': 0.30

In [341]:
# Guardamos en 'best_model_rl' el mejor modelo de n-gramas que hemos conseguido:
best_model_svm = best[best_ch_size]["model"]

# Ajustamos sus umbrales de decisión:
char_th = adjust_thresholds(best_model_svm, B_ch, -2, 2, 300, alg="svm")
_ = predict_with_th(best_model_svm, B_ch, char_th, verbose=True, alg="svm")

cs.SI (0.9%): 0.5819  cs.CL (1.9%): 0.5686  cs.CR (0.8%): 0.3010  cs.NE (1.6%): 0.2876  cs.GR (0.9%): 0.2341  cs.RO (3.3%): 0.2341  eess.IV (3.9%): 0.1940  cs.AI (13.9%): 0.1806  cs.IR (0.4%): 0.1538  physics.chem-ph (0.0%): 0.0334  math.OC (0.8%): 0.0067  cs.LG (52.3%): -0.0201  eess.SP (0.6%): -0.0468  cs.MA (0.7%): -0.0736  cs.CY (0.3%): -0.0870  stat.ML (29.1%): -0.1003  cs.CV (61.3%): -0.1137  cs.SY (0.2%): -0.1271  eess.SY (0.2%): -0.1271  cs.MM (0.6%): -0.1538  cs.IT (0.1%): -0.2074  math.IT (0.1%): -0.2074  math.ST (0.2%): -0.3010  stat.AP (0.1%): -0.3010  stat.TH (0.2%): -0.3010  cs.DC (0.3%): -0.3946  stat.ME (0.1%): -0.6355  cs.HC (0.1%): -0.7559  

=== Métricas con umbrales de decisión optimizados ===
{'accuracy': 0.41219811706917725, 'f1': 0.38773753714702913, 'precision': 0.35872195223338943, 'recall': 0.489705799439699, 'hamming_loss': 0.03268814689199462}


### Resultados del test para SVM - [BORRAR AL ENTREGAR]

In [343]:
vec_test = TfidfVectorizer(analyzer="char_wb", ngram_range=best_ch_size, sublinear_tf=True, min_df=1)
vec_test.fit_transform(train_tok)
X_test = vec_test.transform(test_tok)
X_test.sort_indices()

preds_test = predict_with_th(best_model_svm, X_test, char_th, verbose=False, alg="svm")
print(compute_metrics(y_test, preds_test))

{'accuracy': 0.263050153531218, 'f1': 0.17857268705692245, 'precision': 0.15849332452938464, 'recall': 0.2726237754200775, 'hamming_loss': 0.04266705658722035}
